# Team 02 — Model comparison (Deliverable 4)

**Question:** which model should we carry forward to predict monthly passengers (`OD_PAX`) per route?

**Procedure (same for every candidate):**
1. Read the D3 feature table (`data/processed/team02_features.csv`, made by `feature_set.ipynb`).
2. **Test = the last 12 months.** Train = everything before. Nothing is shuffled.
3. Tree models use the last 6 training months for early stopping, so the test months are never touched while fitting.
4. Every model is scored on the **same test rows** with the **same metrics**: MAE, RMSE, WAPE (lower is better).
   WAPE = total absolute error / total passengers, so it reads as "% of passengers we got wrong".

| Candidate | Owner | Why it is in the comparison |
|---|---|---|
| Same month last year (seasonal naive) | Akshith | the bar every model must beat |
| Ridge on `log1p(OD_PAX)` | Sahith | simple, explainable linear model |
| XGBoost, `log1p` + squared error | Aidan | trees handle many airports + interactions; one side of the loss debate |
| XGBoost, Poisson on raw counts | Aidan | the other side of the loss debate |

Not run: Random Forest, ARIMA, LSTM — reasons in `results/model-comparison.md`.
**NDA:** summaries only; clear outputs before committing.

## 1. Setup and shared helpers

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from xgboost import XGBRegressor

pd.set_option('display.max_columns', 40)
plt.rcParams['figure.figsize'] = (10, 4)
TARGET = 'OD_PAX'
RANDOM_STATE = 42

repo_root = next((f for f in [Path.cwd(), *Path.cwd().parents]
                  if (f / 'DATA.md').is_file() and (f / 'data').is_dir()), None)
if repo_root is None:
    raise FileNotFoundError('Open this notebook from inside the project repository.')
feat_path = repo_root / 'data' / 'processed' / 'team02_features.csv'
if not feat_path.is_file():
    raise FileNotFoundError('Run notebooks/feature_set.ipynb first -- it creates data/processed/team02_features.csv')
df = pd.read_csv(feat_path, parse_dates=['period'])
for c in ['ORG_AIRPRT', 'DES_AIRPRT', 'ORG_CNTRY_CD', 'DES_CNTRY_CD', 'route']:
    df[c] = df[c].astype(str)

CAT = ['ORG_AIRPRT', 'DES_AIRPRT', 'ORG_CNTRY_CD', 'DES_CNTRY_CD']
LAGS = ['pax_same_month_last_year', 'pax_same_month_2_years_ago']
for c in LAGS:
    df['log_' + c] = np.log1p(df[c].clip(lower=0))      # log versions for the linear model

months = np.sort(df['period'].unique())
print(f'feature table loaded: {len(months)} months, {pd.Timestamp(months[0]):%Y-%m} -> {pd.Timestamp(months[-1]):%Y-%m}')

# ---- shared helpers --------------------------------------------------------------------
def metrics(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    return {'MAE': np.mean(np.abs(y - p)), 'RMSE': np.sqrt(np.mean((y - p) ** 2)),
            'WAPE': np.abs(y - p).sum() / np.abs(y).sum()}

def to_xgb(frame, cols, ref):
    """Categoricals as pandas categories with the TRAINING categories only (unseen -> missing)."""
    X = frame[cols].copy()
    for c in CAT:
        if c in cols:
            X[c] = pd.Categorical(X[c].where(X[c].isin(ref[c])), categories=ref[c])
    return X

def fit_xgb(train, cols, loss, val_months=6, **extra):
    """Fit XGBoost with early stopping on the last `val_months` of the training period (never the test months)."""
    ref = {c: sorted(train[c].unique()) for c in CAT}
    cut = np.sort(train['period'].unique())[-val_months]
    fit, val = train[train['period'] < cut], train[train['period'] >= cut]
    if loss == 'poisson':
        obj, yf, yv = 'count:poisson', fit[TARGET], val[TARGET]
    elif loss == 'log1p':
        obj, yf, yv = 'reg:squarederror', np.log1p(fit[TARGET].clip(lower=0)), np.log1p(val[TARGET].clip(lower=0))
    else:
        raise ValueError(loss)
    params = dict(objective=obj, n_estimators=2000, learning_rate=0.05, max_depth=8, min_child_weight=5,
                  subsample=0.8, colsample_bytree=0.8, tree_method='hist', enable_categorical=True,
                  max_cat_to_onehot=1, early_stopping_rounds=100, random_state=RANDOM_STATE)
    params.update(extra)
    m = XGBRegressor(**params)
    m.fit(to_xgb(fit, cols, ref), yf, eval_set=[(to_xgb(val, cols, ref), yv)], verbose=False)
    m.ref_, m.cols_, m.loss_ = ref, cols, loss
    return m

def predict_xgb(m, frame):
    p = m.predict(to_xgb(frame, m.cols_, m.ref_))
    return np.expm1(p).clip(min=0) if m.loss_ == 'log1p' else p.clip(min=0)

def lag_baseline(train, test, lag_col):
    """Seasonal naive: the lag value; if the route has none, the route's training median; else the overall median."""
    route_med = train.groupby('route')[TARGET].median()
    return (test[lag_col].fillna(test['route'].map(route_med)).fillna(train[TARGET].median())).to_numpy()

## 2. The time split

In [ ]:
TEST_MONTHS = 12
cutoff = months[-TEST_MONTHS]
train, test = df[df['period'] < cutoff].copy(), df[df['period'] >= cutoff].copy()
print(f'train: up to {pd.Timestamp(months[-TEST_MONTHS - 1]):%Y-%m} | test: {pd.Timestamp(cutoff):%Y-%m} -> {pd.Timestamp(months[-1]):%Y-%m}')
print(f'test share of rows: {len(test) / len(df):.1%}')

NUM_TREE = ['DEP_MONTH', 'month_sin', 'month_cos', 'months_since_start', 'is_international',
            'pax_same_month_last_year', 'route_seen_last_year', 'pax_same_month_2_years_ago', 'route_seen_2_years_ago']
TREE_COLS = CAT + NUM_TREE
NUM_LIN = ['month_sin', 'month_cos', 'months_since_start', 'is_international',
           'log_pax_same_month_last_year', 'route_seen_last_year', 'log_pax_same_month_2_years_ago', 'route_seen_2_years_ago']
preds = {}

## 3. Baseline — same month last year (Akshith)

In [ ]:
preds['Baseline: same month last year'] = lag_baseline(train, test, 'pax_same_month_last_year')
print(f'test rows that have a real last-year value: {test["pax_same_month_last_year"].notna().mean():.1%} (others use the route median)')

## 4. Ridge on `log1p` (Sahith)

One-hot for airports (airports seen fewer than 24 times are grouped into one "rare" column) and countries, scaled
numeric features, missing lags filled with 0 plus the `route_seen_*` flags. Trained on `log1p(OD_PAX)`, predictions
turned back with `expm1`.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ridge = Pipeline([
    ('prep', ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=24), CAT),
        ('num', make_pipeline(SimpleImputer(strategy='constant', fill_value=0), StandardScaler()), NUM_LIN)])),
    ('m', Ridge(alpha=1.0))])
ridge.fit(train[CAT + NUM_LIN], np.log1p(train[TARGET].clip(lower=0)))
preds['Ridge (log1p)'] = np.expm1(ridge.predict(test[CAT + NUM_LIN])).clip(min=0)

## 5. XGBoost — two losses (Aidan)

Same features and settings; only the loss changes. Airports are native categoricals (no one-hot needed).

In [ ]:
xgb_models = {}
for loss in ['log1p', 'poisson']:
    m = fit_xgb(train, TREE_COLS, loss)
    xgb_models[loss] = m
    preds[f'XGBoost ({loss})'] = predict_xgb(m, test)
    print(f'{loss}: stopped at {m.best_iteration + 1} trees')

## 6. Results — same rows, same metrics

In [ ]:
results = pd.DataFrame({name: metrics(test[TARGET], p) for name, p in preds.items()}).T.sort_values('WAPE')
display(results.round({'MAE': 1, 'RMSE': 1, 'WAPE': 3}))
best = results.index[0]
base = results.loc['Baseline: same month last year', 'WAPE']
print(f'best by WAPE: {best} | improvement over baseline: {(1 - results.loc[best, "WAPE"] / base):.1%}')

best_xgb = 'poisson' if results.loc['XGBoost (poisson)', 'WAPE'] <= results.loc['XGBoost (log1p)', 'WAPE'] else 'log1p'
print(f'loss decision (lower WAPE wins): {best_xgb}  -> set LOSS = "{best_xgb}" in two_year_forecast.ipynb')

## 7. Where the models go wrong — by segment

Errors split by domestic vs international and by route size (small / medium / large routes, by training-period
average). A model that only wins on big routes may still be useless for small-market planning.

In [ ]:
size = train.groupby('route')[TARGET].mean()
test['route_size'] = pd.qcut(test['route'].map(size), 3, labels=['small', 'medium', 'large']).astype(str)
test.loc[test['route'].map(size).isna(), 'route_size'] = 'new route'
test['segment_intl'] = test['is_international'].map({0.0: 'domestic', 1.0: 'international'}).fillna('unknown')

rows = []
for seg_col in ['segment_intl', 'route_size']:
    for seg, idx in test.groupby(seg_col).groups.items():
        mask = test.index.isin(idx)
        row = {'split': seg_col, 'segment': seg, 'share of test passengers': test.loc[mask, TARGET].sum() / test[TARGET].sum()}
        for name, p in preds.items():
            row[name] = metrics(test.loc[mask, TARGET], p[mask])['WAPE']
        rows.append(row)
display(pd.DataFrame(rows).set_index(['split', 'segment']).round(3))

## 8. What the best tree model relies on

In [ ]:
m = xgb_models[best_xgb]
imp = pd.Series(m.get_booster().get_score(importance_type='gain')).reindex(TREE_COLS).fillna(0)
(imp / imp.sum()).sort_values().plot.barh(title=f'XGBoost ({best_xgb}) — share of total gain')
plt.xlabel('share of gain'); plt.tight_layout(); plt.show()

## 9. Tuning — hyperparameter search and regularisation

Both XGBoost fits above used the full 2,000-tree budget, so they never finished learning. Here we search the
settings of the winning loss.

- **No test peeking:** each setting is scored on the **last 12 training months** (fit on the months before them).
  The test year is only used once, at the end, for the chosen setting.
- **What is searched:** learning rate, tree depth, and the regularisation knobs `min_child_weight` (minimum data per
  leaf) and `reg_lambda` (L2 penalty on leaf values). For Poisson also `max_delta_step`, which caps how far one tree
  can move a prediction (the default 0.7 makes Poisson learn slowly on large counts).
- A random sample of `N_TRIALS` settings plus the current settings. Each trial prints its time; lower `N_TRIALS`
  if it is too slow.

In [ ]:
import itertools, time

N_TRIALS = 8
CAP = 4000                                    # tree budget during the search (early stopping still applies)
tr_months = np.sort(train['period'].unique())
tune_cut = tr_months[-12]
tune_fit, tune_val = train[train['period'] < tune_cut], train[train['period'] >= tune_cut]

GRID = {'learning_rate': [0.05, 0.1, 0.2], 'max_depth': [4, 6, 8], 'min_child_weight': [5, 20],
        'reg_lambda': [1, 10]}
CURRENT = {'learning_rate': 0.05, 'max_depth': 8, 'min_child_weight': 5, 'reg_lambda': 1}
if best_xgb == 'poisson':
    GRID['max_delta_step'] = [0.7, 2.0]
    CURRENT['max_delta_step'] = 0.7
combos = [dict(zip(GRID, v)) for v in itertools.product(*GRID.values())]
rng = np.random.default_rng(RANDOM_STATE)
trials = [CURRENT] + [combos[i] for i in rng.choice(len(combos), N_TRIALS, replace=False) if combos[i] != CURRENT]

val_base = metrics(tune_val[TARGET], lag_baseline(tune_fit, tune_val, 'pax_same_month_last_year'))['WAPE']
rows, t0 = [], time.time()
for i, p in enumerate(trials, 1):
    m = fit_xgb(tune_fit, TREE_COLS, best_xgb, n_estimators=CAP, **p)
    rows.append({**p, 'trees': m.best_iteration + 1, 'still improving at cap': m.best_iteration + 1 > CAP - 100,
                 'val WAPE': metrics(tune_val[TARGET], predict_xgb(m, tune_val))['WAPE']})
    print(f'trial {i}/{len(trials)} done -- {time.time() - t0:.0f}s so far')

tuning = pd.DataFrame(rows).sort_values('val WAPE').reset_index(drop=True)
display(tuning.round(3))
print(f'baseline on the same validation months: WAPE {val_base:.3f}')
best_p = tuning.iloc[0][list(GRID)].to_dict()
TUNED = {k: int(v) if k == 'max_depth' else float(v) for k, v in best_p.items()}
TUNED['n_estimators'] = CAP
print('chosen settings:', TUNED)

In [ ]:
# Refit the chosen setting on the full training period and score it ONCE on the test year
tuned_m = fit_xgb(train, TREE_COLS, best_xgb, **TUNED)
preds[f'XGBoost ({best_xgb}, tuned)'] = predict_xgb(tuned_m, test)
print(f'tuned model: {tuned_m.best_iteration + 1} trees (cap {CAP})')

results = pd.DataFrame({name: metrics(test[TARGET], p) for name, p in preds.items()}).T.sort_values('WAPE')
results['WAPE vs baseline'] = results['WAPE'] / base - 1
display(results.round({'MAE': 1, 'RMSE': 1, 'WAPE': 3, 'WAPE vs baseline': 3}))
print(f'best by WAPE: {results.index[0]}')
print(f'for two_year_forecast.ipynb -> LOSS = "{best_xgb}"  and  TUNED = {TUNED}')

## 10. Learning curves

**(a) Error vs number of trees.** Validation loss (last 6 training months) as trees are added, current vs tuned
settings. A curve still falling at the end means the model was stopped too early.

**(b) Error vs amount of history.** The tuned model trained on only the most recent 2, 3, 4 years (and all) of the
search period, scored on the same validation months. If error keeps dropping with more years, more history helps;
if it flattens or rises, old years (e.g. 2020) add noise.

In [ ]:
VIS = repo_root / 'data' / 'visuals'
VIS.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots()
for label, mdl in [('current settings', xgb_models[best_xgb]), ('tuned settings', tuned_m)]:
    hist = mdl.evals_result()['validation_0']
    metric_name = next(iter(hist))
    ax.plot(hist[metric_name], label=f'{label} (best at {mdl.best_iteration + 1})')
ax.set_xlabel('number of trees'); ax.set_ylabel(f'validation {metric_name}'); ax.legend()
ax.set_title(f'XGBoost ({best_xgb}) — validation loss vs trees')
fig.savefig(VIS / 'd4_learning_curve_trees.png', dpi=150, bbox_inches='tight'); plt.show()

fit_months = np.sort(tune_fit['period'].unique())
sizes = [n for n in [24, 36, 48] if n < len(fit_months)] + [len(fit_months)]
curve = []
for n in sizes:
    sub = tune_fit[tune_fit['period'] >= fit_months[-n]]
    m = fit_xgb(sub, TREE_COLS, best_xgb, **TUNED)
    curve.append({'years of history': round(n / 12, 1),
                  'val WAPE': metrics(tune_val[TARGET], predict_xgb(m, tune_val))['WAPE']})
curve = pd.DataFrame(curve)
display(curve.round(3))
fig, ax = plt.subplots()
ax.plot(curve['years of history'], curve['val WAPE'], marker='o', label='tuned XGBoost')
ax.axhline(val_base, color='grey', ls='--', label='baseline (same month last year)')
ax.set_xlabel('years of training history'); ax.set_ylabel('validation WAPE'); ax.legend()
ax.set_title('Learning curve — error vs amount of history')
fig.savefig(VIS / 'd4_learning_curve_history.png', dpi=150, bbox_inches='tight'); plt.show()

## 11. Conclusions (final run, Nov 22, 2026 — rounded)

- **Best model:** the seasonal baseline (same month last year), WAPE about 18.0%. XGBoost Poisson came close at
  about 18.9% (about 5% worse), XGBoost `log1p` about 19.3%. No model beat the baseline on this test year.
- **Loss decision (log1p vs Poisson):** Poisson. Lower MAE, RMSE and WAPE than `log1p` (Aidan).
- **Ridge vs trees:** Ridge on `log1p` scored about 82% WAPE, about 4.6 times the baseline (Sahith; fixed
  `alpha=1.0`, not tuned). A linear model with one-hot airports cannot build the month x route interactions the
  trees find, so it is far behind in every segment.
- **Segments where every model struggles:** international routes (about 24–25% WAPE vs about 16–17% domestic) and
  small routes (about 36–40% vs about 17–18% for large routes). Large routes carry about 89% of passengers, so the
  headline WAPE mostly reflects them.
- **What the trees rely on:** same month last year (about half the gain) and same month two years ago (about a
  quarter). The model mostly re-learns the seasonal naive, which is why it lands near the baseline but not below it.
- **Tuning:** the search in section 9 is built and tested, but the run did not finish before the deadline, so no
  tuned result is reported. Both original XGBoost fits used the full 2,000-tree budget (also noted by Sahith in
  `results/forecast-evaluation.md`), so finishing this search is the first next step.
- **Learning curves:** depend on section 9; not run yet.
- **Carried into D5:** XGBoost Poisson with the original settings (`TUNED = {}` in `two_year_forecast.ipynb`).

### Before committing
Kernel → Restart & Clear Output, then save.
